In [ ]:
suppressPackageStartupMessages(library(Seurat))
library(ggplot2)
library(dplyr)
library(ggpubr)

In [ ]:
main_path <- "/expanse/lustre/projects/csd728/m7huang/single_cell_paik/samples_2/"

In [ ]:
data_dir  <-  "/expanse/lustre/projects/csd728/m7huang/single_cell_paik/outputs_20260919/DP_only/"

# Load data

In [ ]:
meta <- read.table("/expanse/lustre/projects/csd728/m7huang/single_cell_paik/data/20260919_SraRunTable.csv", sep = ",", header = TRUE)
head(meta)

In [ ]:
files <- meta$Run
files

In [ ]:
# Extract names of files needed for creating expression matrix per sample
sample_list <- list()

for (sample in files){
    mat <- paste0(main_path, sample, "/Solo.out/Gene/filtered/matrix.mtx")
    feat <- paste0(main_path, sample, "/Solo.out/Gene/filtered/features.tsv")
    barcodes <- paste0(main_path, sample, "/Solo.out/Gene/filtered/barcodes.tsv")
    sample_list[[sample]] <- c(mat, feat, barcodes)
}

In [ ]:
seurat_list <- list()

for (sample in names(sample_list)){
    # for creating the Seurat object from STARSolo output
    expression_matrix <- ReadMtx(mtx = sample_list[[sample]][1],
                                 features = sample_list[[sample]][2],
                                 cells = sample_list[[sample]][3])
    seurat_object <- CreateSeuratObject(counts = expression_matrix)
    seurat_list[[sample]] <- seurat_object
}

In [ ]:
saveRDS(seurat_list, 
     file=file.path(paste0(data_dir, '/seurat_list.rds')))

# QC
"cells with fewer than 200 detected genes, fewer than 1000 UMI counts, cells with the total mitochondrial gene expression exceeded 20%, and cells with more than 7000 genes expressed per cell were removed"

## Filtration

In [ ]:
for (num in 1:length(seurat_list)){
    seurat_list[[num]]@meta.data$orig.ident <- files[num]

    obj <- seurat_list[[num]]
    obj <- PercentageFeatureSet(obj, "^MT-", col.name = "percent.mt", assay='RNA')
    seurat_list[[num]] <- obj
}

In [ ]:
pdf(paste0(data_dir, 'qc.pdf'), width = 20, height = 15)

for (obj in seurat_list){ 
     Idents(obj) <- obj@meta.data$orig.ident

     p1 <- VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3)
     p2 <- FeatureScatter(obj, feature1 = "nCount_RNA", feature2 = "percent.mt")
     p3 <- FeatureScatter(obj, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")

     print(p1 / (p2 + p3  ))
}

dev.off()

In [ ]:
sn_objects_filtered <- list()

for (obj in seurat_list){ 
    name <- obj@meta.data$orig.ident[1]

    print(paste0(name, " - Original: ", nrow(obj@meta.data)))
    obj <- subset(obj, subset = nCount_RNA>1000 & nCount_RNA<50000 & 
                                nFeature_RNA>200 & nFeature_RNA<7000 & 
                                percent.mt<20)
    
    print(paste0(name, " - Filtered: ", nrow(obj@meta.data)))
    sn_objects_filtered[[name]] <- obj
}

In [ ]:
saveRDS(sn_objects_filtered, 
     file=file.path(paste0(data_dir, '/sn_objects_filtered.rds')))

## Doublet

In [ ]:
sn_objects_filtered <- readRDS(file.path(paste0(data_dir, '/sn_objects_filtered.rds')))

In [ ]:
suppressPackageStartupMessages(library(DoubletFinder))

In [ ]:
seurat_pre_processing <- function(obj){
    DefaultAssay(obj) <- "RNA"
    obj <- NormalizeData(obj)
    obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 2000)
    obj <- ScaleData(obj)
    
    obj <- RunPCA(obj, npcs = 10)
    obj <- RunUMAP(obj, dims = 1:10)
    obj <- FindNeighbors(obj, dims = 1:10)
    
    ## only use FindClusters after the correct resolution is determined. 
    obj <- FindClusters(obj, resolution = 0.5)
    obj@meta.data$RNA_snn_res.0.5 <- as.character(obj@meta.data$RNA_snn_res.0.5)   
    
    obj
}

In [ ]:
sn_objects_pre_process <- list()

for (obj in sn_objects_filtered){ 
    obj <- seurat_pre_processing(obj) 
    name <- obj@meta.data$orig.ident[1]
    
    print(name)
    sn_objects_pre_process[[name]] <- obj
}

In [ ]:
# options(repr.plot.width=5, repr.plot.height=5)

# for (obj in sn_objects_pre_process){
#     name <- obj@meta.data$orig.ident[1]
#     p1 <- DimPlot(obj, group.by = 'RNA_snn_res.0.5')
#     print(p1)
# }

In [ ]:
# doubletFinder
doublet_finder <- function(obj){
    obj@meta.data$RNA_snn_res.0.5 <- as.character(obj@meta.data$RNA_snn_res.0.5)   

    ## pK Identification (no/yes ground-truth)
    sweep.obj <- paramSweep(obj, PCs = 1:10, sct = FALSE)
    sweep.stats_obj <- summarizeSweep(sweep.obj, GT = FALSE)
    bcmvn_obj <- find.pK(sweep.stats_obj)
    pk <- as.numeric(as.character(bcmvn_obj$pK[which.max(bcmvn_obj$BCmetric)]))
    
    ## estimate doublet formation rate based on the number of cells 
    name <- obj@meta.data$orig.ident[1]
    mypath <- file.path(paste0('/expanse/lustre/projects/csd728/m7huang/single_cell_paik/samples_2/',
                               name, '/Solo.out/Gene/Summary.csv'))
    num_cells <- as.integer(subset(read.csv(mypath, header = FALSE), V1 == 'Estimated Number of Cells')$V2)
    print(paste0(name, ": ", num_cells))
    doublet_formation_rate <- read.csv('/expanse/lustre/projects/csd834/m7huang/multiome/tools/doublet_formation_rates.csv') 
    doublet_formation <- doublet_formation_rate$multiplet_rate[which.min(abs(doublet_formation_rate$num_cells_recovered - num_cells))]
    
    ## Homotypic Doublet Proportion Estimate for high and low stringent
    homotypic.prop <- modelHomotypic(obj@meta.data$RNA_snn_res.0.5)
    nExp_poi <- round(doublet_formation * nrow(obj@meta.data))
    nExp_poi.adj <- round(nExp_poi*(1-homotypic.prop))
    
    ## doublet finder only works with metadata as character - double check!
    ## doublet finder with high and low classification stringency
    obj@meta.data$RNA_snn_res.0.5 <- as.character(obj@meta.data$RNA_snn_res.0.5)
    obj <- doubletFinder(obj, PCs=1:10, pK=pk, nExp=nExp_poi,
                     annotation=obj@meta.data$RNA_snn_res.0.5, sct = FALSE)
    pANN.heterotypic <- colnames(obj@meta.data)[grepl("^pANN", colnames(obj@meta.data))]
    colnames(obj@meta.data)[grepl(paste0(sub("pANN_", "", pANN.heterotypic), "$"), colnames(obj@meta.data)) &
            colnames(obj@meta.data) != pANN.heterotypic] <- "DF_hetero_high_conf"
    obj <- doubletFinder(obj, PCs=1:10, pK=pk, nExp=nExp_poi.adj, 
                         sct=FALSE, annotation=obj@meta.data$RNA_snn_res.0.5, reuse.pANN=pANN.heterotypic)
    colnames(obj@meta.data)[grepl("^DF\\.classifications_[0-9]", colnames(obj@meta.data))] <- "DF_homo_low_conf"
    obj
}

In [ ]:
sn_objects_doubletFinder <- list()

for (obj in sn_objects_pre_process){ 
    obj <- doublet_finder(obj) 
    name <- obj@meta.data$orig.ident[1]
    sn_objects_doubletFinder[[name]] <- obj
}

In [ ]:
# filter out predicted doublets 
sn_objects_doubletFinder_filtered <- list()

for (obj in sn_objects_doubletFinder){ 
    obj <- subset(x = obj, subset = DF_hetero_high_conf == "Singlet" & DF_homo_low_conf == "Singlet")
    name <- obj@meta.data$orig.ident[1]

    sn_objects_doubletFinder_filtered[[name]] <- obj
    
    print(paste0(obj@meta.data$orig.ident[1], ": ", nrow(obj@meta.data)))
}

In [ ]:
saveRDS(sn_objects_doubletFinder_filtered, 
     file=file.path(paste0(data_dir, '/sn_objects_doubletFinder_filtered.rds')))

## Ambient RNA

In [ ]:
sn_objects_doubletFinder_filtered <- readRDS(file.path(paste0(data_dir, '/sn_objects_doubletFinder_filtered.rds')))

In [ ]:
library(SoupX)
library(ggplot2)

In [ ]:
run_soupx <- function(sc_obj){
    name <- sc_obj@meta.data$orig.ident[1]
    tod <- ReadMtx(mtx = paste0(main_path, name, "/Solo.out/Gene/raw/matrix.mtx"),
                                 features = paste0(main_path, name, "/Solo.out/Gene/raw/features.tsv"),
                                 cells =paste0(main_path, name, "/Solo.out/Gene/raw/barcodes.tsv"))

    
   
    filt_GEX <- sc_obj@assays$RNA@layers$counts
    colnames(filt_GEX) <- rownames(sc_obj@assays$RNA@cells)
    rownames(filt_GEX) <- rownames(sc_obj@assays$RNA@features)  

    sc = SoupChannel(tod, filt_GEX, calcSoupProfile = FALSE)
    sc = estimateSoup(sc)

    clusters <- data.frame(
      Barcode = rownames(sc_obj@meta.data),
      Cluster = sc_obj@meta.data[["RNA_snn_res.0.5"]]
    )

    ## reorder to match the order of filt_GEX column names
    ## add cluster data
    clusters <- clusters[match(colnames(filt_GEX), clusters$Barcode), ]
    sc <- setClusters(sc, setNames(clusters$Cluster, rownames(clusters)))

    sc <- setDR(sc, Embeddings(sc_obj, reduction = "umap")[colnames(sc$toc), c("umap_1", "umap_2")])

    sc = autoEstCont(sc)
    out = adjustCounts(sc)  
    return(list(sc = sc, out = out))
}

plot_change <- function(name, sc, out){
    options(repr.plot.width=15, repr.plot.height=5)

    p1 <- plotChangeMap(sc, out, "ITGA5") + ggtitle(paste0("ITGA5 - ", name, " (SoupX)"))
    p2 <- plotChangeMap(sc, out, "ITGA6") + ggtitle(paste0("ITGA6 - ", name, " (SoupX)"))
    p3 <- plotChangeMap(sc, out, "VIM") + ggtitle(paste0("VIM - ", name, " (SoupX)"))

    print(p1 | p2 | p3)
}

In [ ]:
pdf(paste0(data_dir, 'soupx_change_by_gene.pdf'), width = 15, height = 5)

soupx_objects <- list()

for (obj in sn_objects_doubletFinder_filtered){
    name <- obj@meta.data$orig.ident[1]

    soupx_objects[[name]] <- run_soupx(obj)

    plot_change(name, soupx_objects[[name]]$sc, soupx_objects[[name]]$out)
}

dev.off()

In [ ]:
saveRDS(soupx_objects, 
     file=file.path(paste0(data_dir, 'soupx_objects.RData')))

In [ ]:
sn_objects_soupx_filtered <- list()

for (obj in sn_objects_doubletFinder_filtered){
    name <- obj@meta.data$orig.ident[1]

    obj[['RNA']] <- CreateAssay5Object(counts = soupx_objects[[name]]$out)
    sn_objects_soupx_filtered[[name]] <- obj
}

In [ ]:
# Merge all seurat objects
paik.combined <- merge(sn_objects_soupx_filtered[[1]], 
                         y = sn_objects_soupx_filtered[2:length(sn_objects_soupx_filtered)], 
                         add.cell.ids = (names(sn_objects_soupx_filtered)), project = "paik")
paik.combined

In [ ]:
saveRDS(paik.combined, 
     file=file.path(paste0(data_dir, 'paik.combined.rds')))

## Get cell barcodes

In [ ]:
# folder <- list.files(main_path)
# for (samp_name in folder){
#     samp <- subset(paik.combined, orig.ident == samp_name)
#     samp <- RenameCells(samp, new.names = gsub(paste0(samp_name, "_"), "", 
#                                                Cells(samp)))
#     print(length(Cells(samp)))
#     write.table(as.data.frame(Cells(samp)), 
#                 paste0("/expanse/lustre/projects/csd728/m7huang/single_cell_paik/cell_barcodes/", 
#                          samp_name, "/", samp_name, "_all_cells.barcodes.txt"), 
#                 quote = FALSE, col.names = FALSE, row.names = FALSE)
# }

# Annotation

## pre-processing

In [ ]:
combined <- readRDS(file.path(paste0(data_dir, 'paik.combined.rds')))

In [ ]:
# identify which to exclude
meta <- read.table("/expanse/lustre/projects/csd728/m7huang/single_cell_paik/data/20260919_SraRunTable.csv", sep = ",", header = TRUE)

# QC issues 
meta_keep <- subset(meta, gestational_week != "39+6")

meta_keep <- subset(meta_keep, source_name == "Placenta (Deep placenta)")

In [ ]:
meta_keep$Run

In [ ]:
length(unique(meta_keep$BioSample))

In [ ]:
combined <- subset(combined, subset = orig.ident %in% meta_keep$Run)

In [ ]:
length(unique(combined$orig.ident))

In [ ]:
unique(combined$orig.ident)

In [ ]:
DefaultAssay(combined) <- 'RNA'

combined <- NormalizeData(combined)
combined <- FindVariableFeatures(combined, selection.method = "vst", nfeatures = 2000)
combined <- ScaleData(combined)

combined <- RunPCA(combined, npcs = 10)
combined <- RunUMAP(combined, dims = 1:10, reduction.name = "umap.rna")
combined <- FindNeighbors(combined, dims = 1:10, verbose = TRUE)
combined <- FindClusters(combined, verbose = TRUE, resolution = 0.1)

In [ ]:
options(repr.plot.width=16, repr.plot.height=5)

DimPlot(combined,  group.by = c("orig.ident", "RNA_snn_res.0.1"))

In [ ]:
combined <- IntegrateLayers(
  object = combined, method = HarmonyIntegration,
  orig.reduction = "pca", new.reduction = "harmony",
  verbose = FALSE
)

In [ ]:
combined <- FindNeighbors(combined, reduction = "harmony", dims = 1:10)
combined <- FindClusters(combined, resolution = 0.1, cluster.name = "harmony_clusters")

In [ ]:
combined <- RunUMAP(combined, reduction = "harmony", dims = 1:10, reduction.name = "umap.harmony")

In [ ]:
DimPlot(combined, reduction = "umap.harmony", group.by = c("orig.ident", "RNA_snn_res.0.1"))

In [ ]:
combined <- JoinLayers(combined)
combined

In [ ]:
keep_cols <- c("orig.ident", "nCount_RNA", "nFeature_RNA", "percent.mt", "RNA_snn_res.0.5", 
              "RNA_snn_res.0.1", "seurat_clusters", "harmony_clusters", "DF_homo_low_conf",
              "DF_hetero_high_conf")

combined@meta.data <- combined@meta.data[, keep_cols, drop = FALSE]

In [ ]:
data_dir  <-  "/expanse/lustre/projects/csd728/m7huang/single_cell_paik/outputs_20260919/DP_only/"

In [ ]:
saveRDS(combined, 
     file=file.path(paste0(data_dir, 'DP_onlypaik.harmony.combined.20260920.rds')))

## check features

In [ ]:
combined <- readRDS(file=file.path(paste0(data_dir, 'DP_onlypaik.harmony.combined.20260920.rds')))

In [ ]:
meta <- read.table("/expanse/lustre/projects/csd728/m7huang/single_cell_paik/data/20260919_SraRunTable.csv", sep = ",", header = TRUE)
colnames(meta)[colnames(meta) == "Run"] <- "orig.ident"
colnames(meta)

In [ ]:
table(meta$gestational_week)

In [ ]:
matched <- meta[match(combined@meta.data$orig.ident, meta$orig.ident), ]
combined <- AddMetaData(combined, metadata = matched)

In [ ]:
combined$clusters <- Idents(combined)
df <- combined[[]]

In [ ]:
table(df[c("clusters", "orig.ident")])

In [ ]:
table(df[c("clusters", "source_name")])

In [ ]:
options(repr.plot.width=15, repr.plot.height=5)

DimPlot(combined, reduction = "umap.harmony", group.by = c("harmony_clusters", "source_name"), label = TRUE)

In [ ]:
library(dplyr)

combined@meta.data %>%
  group_by(seurat_clusters) %>%
  summarise(orig_idents = paste(unique(orig.ident), collapse = ", "))

In [ ]:
genes <- c("ACTA2","PECAM1","IDO1", "CLDN5",
          "CD74","COL15A1","MKI67",
           "HLA-A","HLA-B","HLA-C","COL1A1", "ITGA6", "ITGA5", "CD9", "VIM")

trophoblasts <- c("ASCL2","HLA-G", "KRT14", "CDH1", "EGFR", "KRT5",
                  "KRT6A","KRT7","KRT8", "KRT10","KRT15", "KRT16","KRT19")

immune <- c("LYVE1","NCAM1","MS4A1",
            "CD4","CD8A","CD68","PTPRC","FUT4",
            "TPSAB1","CD3E")

ctb <- c("BCAM", "SLC6A4", "PARP1", "EGFR", "TP63", "SLC27A2", "TENM3")
evt <- c("HLA-G", "ASCL2", "ITGA5", "FN1", "LAIR2", "NOTUM", "LPCAT1", "PRG2", "AOC1")
pre_stb <- c("CYP19A1", "ERVV-1", "GREM2", "ERVFRD-1", "LGALS16", "EPS8L1", "SERPINB2") 

In [ ]:
options(repr.plot.width=20, repr.plot.height=20)

FeaturePlot(combined, features = trophoblasts, reduction = "umap.harmony", raster = TRUE)


In [ ]:
options(repr.plot.width=20, repr.plot.height=15)

FeaturePlot(combined, features = genes, reduction = "umap.harmony", raster = TRUE)


In [ ]:
options(repr.plot.width=20, repr.plot.height=10)

FeaturePlot(combined, features = immune, reduction = "umap.harmony", raster = TRUE)


In [ ]:
options(repr.plot.width=20, repr.plot.height=20)

FeaturePlot(combined, features = ctb, reduction = "umap.harmony", raster = TRUE)


In [ ]:
options(repr.plot.width=20, repr.plot.height=15)

FeaturePlot(combined, features = evt, reduction = "umap.harmony", raster = TRUE)


In [ ]:
options(repr.plot.width=20, repr.plot.height=15)

FeaturePlot(combined, features = pre_stb, reduction = "umap.harmony", raster = TRUE)


In [ ]:
new_labels <- c(
    "0" = "Immune",
    "1" = "Immune",
    "2" = "Immune",
    "3" = "Immune",
    "4" = "Fibroblast",
    "5" = "Cytotrophoblast",
    "6" = "Extravillous trophoblast",
    "7" = "Immune",
    "8" = "Endothelial",
    "9" = "Immune",
    "11" = "Immune",
    "10" = "Fibroblast"
)
combined1 <- RenameIdents(combined, new_labels)
combined1$cell_types <- Idents(combined1)

In [ ]:
options(repr.plot.width=8, repr.plot.height=6)
cols <- c(
  "Immune" = "#8FA8C7",
  "Endothelial" = "#5B6770",
  "Fibroblast" = "#D6B47C",
  "Cytotrophoblast" = "#B8A1C8",
  "Extravillous trophoblast" = "#C99A9A"
)

tiff(paste0(data_dir, "final/all_umap.tiff"), units="in", 
     width=9, height=6, res=300)
DimPlot(combined1, reduction = "umap.harmony", raster = FALSE, group.by = "cell_types", cols = cols) & labs(x = "UMAP1", y = "UMAP2") & 
ggtitle("")
dev.off()

In [ ]:
DimPlot(combined1, reduction = "umap.harmony", raster = FALSE, group.by = "cell_types", cols = cols)

In [ ]:
options(repr.plot.width=12, repr.plot.height=5)

DimPlot(combined1, split.by = "sample_description", reduction = "umap.harmony", raster = FALSE) 


In [ ]:
df <- combined1[[]]

df$disease <- ifelse(df$sample_description == "full term birh", "NTT-N",
                 ifelse(df$sample_description == "preterm birth", "EP-PE", df$sample_description))


df2 <- as.data.frame(table(df[c("cell_types", "disease")]))
write.csv(df2, paste0(data_dir, "final/cell_counts_by_disease.csv"))

df2 <- as.data.frame(table(df[c("cell_types", "Sample.Name")]))
write.csv(df2, paste0(data_dir, "final/cell_counts_by_sample.csv"))

# Subset trophoblast

## Reprocess sample

In [ ]:
combined <- readRDS(file=file.path(paste0(data_dir, 'DP_onlypaik.harmony.combined.20260920.rds')))

In [ ]:
troph <- subset(combined, harmony_clusters %in% c(5, 6))
troph[["RNA"]] <- split(troph[["RNA"]], f = troph$orig.ident)

In [ ]:
DefaultAssay(troph) <- 'RNA'

troph <- NormalizeData(troph)
troph <- FindVariableFeatures(troph, selection.method = "vst", nfeatures = 2000)
troph <- ScaleData(troph)

troph <- RunPCA(troph, npcs = 10)
troph <- RunUMAP(troph, dims = 1:10, reduction.name = "umap.rna")
troph <- FindNeighbors(troph, dims = 1:10, verbose = TRUE)

troph <- FindClusters(troph, verbose = TRUE, resolution = 0.1)

In [ ]:
troph <- IntegrateLayers(
    object = troph, method = HarmonyIntegration,
    orig.reduction = "pca", new.reduction = "harmony",  
    group.by.vars = "orig.ident", 
  verbose = FALSE
)

In [ ]:
# combined <- FindNeighbors(combined, reduction = "harmony", dims = 1:10)
troph <- FindClusters(troph, resolution = 0.1, cluster.name = "harmony_troph_clusters")

In [ ]:
troph <- RunUMAP(troph, reduction = "harmony", dims = 1:10, reduction.name = "umap.harmony")


In [ ]:
DimPlot(troph, reduction = "umap.harmony", 
        group.by = c("orig.ident", "harmony_clusters"))

In [ ]:
troph <- JoinLayers(troph)
troph

In [ ]:
troph$cell_type <- ifelse(troph$harmony_clusters == 5, "CTB", "EVT")


In [ ]:
saveRDS(troph, 
     file=file.path(paste0(data_dir, 'DP_onlypaik.troph.harmony.combined.rds')))

## Visualization + loading (trophoblast)

In [ ]:
troph <- readRDS(file.path(paste0(data_dir, 'DP_onlypaik.troph.harmony.combined.rds')))

In [ ]:
meta <- read.table("/expanse/lustre/projects/csd728/m7huang/single_cell_paik/data/20260919_SraRunTable.csv", sep = ",", header = TRUE)
colnames(meta)[colnames(meta) == "Run"] <- "orig.ident"
colnames(meta)

In [ ]:
matched <- meta[match(troph@meta.data$orig.ident, meta$orig.ident), ]
troph <- AddMetaData(troph, metadata = matched)

In [ ]:
troph$disease <- ifelse(troph$sample_description == "full term birh", "NTT-N",
                 ifelse(troph$sample_description == "preterm birth", "EP-PE", troph$sample_description))
troph$source <- ifelse(troph$source_name == "Placenta (Deep placenta)", "Deep placenta",
                 ifelse(troph$source_name == "Placenta (Maternal - Fetal interface)", "Maternal-fetal interface",
                        troph$source_name))


In [ ]:
options(repr.plot.width=12, repr.plot.height=10)
DimPlot(troph, reduction = "umap.harmony", 
        group.by = c("disease", "offspring_sex", 
                     "cell_type", "source"))

In [ ]:
group = "Sample.Name"
tiff(paste0(data_dir, "final/troph", group, "_umap.tiff"), units="in", 
         width=7, height=6, res=300)
DimPlot(troph, reduction = "umap.harmony", 
        group.by = group) + ggtitle("")
dev.off()

In [ ]:
group = "cell_type"
cols <- c(
  "CTB" = "#B8A1C8",
  "EVT" = "#C99A9A"
)

tiff(paste0(data_dir, "final/troph", group, "_umap.tiff"), units="in", 
         width=6, height=6, res=300)
DimPlot(troph, reduction = "umap.harmony", 
        group.by = group, cols = cols) & NoLegend() & ggtitle("")
dev.off()

In [ ]:
options(repr.plot.width=5, repr.plot.height=5)

DimPlot(troph, reduction = "umap.harmony", 
        group.by = group, cols = cols) & NoLegend() & ggtitle("")

In [ ]:
options(repr.plot.width=10, repr.plot.height=5)

FeaturePlot(troph, reduction = "umap.harmony", features = c("HLA-G", "KRT7"))

In [ ]:
options(repr.plot.width=6, repr.plot.height=6)
for (group in c("disease", "offspring_sex",  "source"))
{
    tiff(paste0(data_dir, "final/troph", group, "_umap.tiff"), units="in", 
         width=6, height=6, res=300)
    p <- DimPlot(troph, reduction = "umap.harmony", group.by = group)
    print(p & labs(x = "UMAP1", y = "UMAP2") & NoLegend() & ggtitle(""))
    dev.off()
}

In [ ]:
options(repr.plot.width=6, repr.plot.height=6)

for (group in c("EGFR", "HLA-G", "KRT7"))
{
    tiff(paste0(data_dir, "final/troph", group, "_umap.tiff"), units="in", 
         width=6, height=6, res=300)
    p <- FeaturePlot(troph, reduction = "umap.harmony", features = group)
    print(p & labs(x = "UMAP1", y = "UMAP2") & NoLegend() & ggtitle(""))
    dev.off()
}

In [ ]:
Idents(troph) <- "cell_type"

# Define an order of cluster identities
my_levels <- c("CTB", "EVT")

# Relevel object@ident
Idents(troph) <- factor(x = Idents(troph), levels = my_levels)

In [ ]:
options(repr.plot.width=15, repr.plot.height=6)
cols <- c(
  "CTB" = "#B8A1C8",
  "EVT" = "#C99A9A"
)
VlnPlot(troph, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), alpha = 0.25,
        ncol = 3, cols = c('#B8A1C8', '#C99A9A')) & ggtitle("") 

In [ ]:
tiff(paste0(data_dir, "final/troph_qc_umap.tiff"), units="in", 
          width=16, height=6, res=300)
VlnPlot(troph, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), alpha = 0.25,
        ncol = 3, cols = c('#B8A1C8', '#C99A9A')) & ggtitle("") 
dev.off()

# UCell (trophoblast)

In [ ]:
library(UCell)
library("ggpubr")

In [ ]:
hypoxia_genes <- list(
  "ACOT7", "ADM", "AK4", "ALDOA", "ANKRD37", "ANLN", "BNIP3", "MRGBP",
  "CA9", "CDKN3", "CHCHD2", "CORO1C", "CTSV", "DDIT4", "ENO1", "ESRP1",
  "GAPDH", "GPI", "HILPDA", "HK2", "KIF20A", "KIF4A", "LDHA", "LRRC42",
  "MAD2L2", "MAP7D1", "MCTS1", "MIF", "MRPL13", "MRPL15", "MRPS17",
  "NDRG1", "PNP", "P4HA1", "PFKP", "PGAM1", "PGK1", "PSMA7", "PSRC1",
  "SEC61G", "SHCBP1", "SLC16A1", "SLC25A32", "SLC2A1", "TPI1", "TUBA1B",
  "TUBA1C", "TUBB6", "UTP11", "VEGFA", "YKT6")

In [ ]:
signatures <- list(hypoxia_genes)

In [ ]:
troph1 <- AddModuleScore_UCell(troph, 
                                      features=signatures, name=NULL)

In [ ]:
# write.csv(troph1[[]], paste0(data_dir, "meta/troph_UCell_20260920.csv"))
# saveRDS(troph1, 
#         file=file.path(paste0(data_dir, 'DP_onlypaik.UCellA.troph1.20260920.rds')))

# infercnv (trophoblast)

In [ ]:
library("infercnv")

In [ ]:
setwd("/expanse/lustre/projects/csd728/m7huang/single_cell_paik")
data_dir  <-  "/expanse/lustre/projects/csd728/m7huang/single_cell_paik/outputs_20260919/DP_only/"
troph <- readRDS(file=file.path(paste0(data_dir, 'DP_onlypaik.UCell.troph1.20260920.rds')))
out_dir = "/expanse/lustre/projects/csd728/m7huang/single_cell_paik/infer_cnv/infer_cnv_files_hmm/"

In [ ]:
# # 4 threads by default

# meta <- troph@meta.data
# meta$disease2 <- ifelse(meta$sample_description == "full term birh", "normal",
#                  ifelse(meta$sample_description == "preterm birth", "PePT", meta$sample_description))
# meta$ct_disease <- paste(meta$disease2, meta$cell_type, sep = "_")
# meta$barcode <- (rownames(meta))
# head(meta)

# meta <- meta[c("barcode", "ct_disease")]
# sample = "troph"
# write.table(meta, paste0("infer_cnv/", sample, "_infer_cnv.tsv"), sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)

# counts_matrix = LayerData(troph, assay = "RNA", layer = "counts")
# infercnv_obj = CreateInfercnvObject(raw_counts_matrix= counts_matrix,
#                                     annotations_file=paste0("infer_cnv/", sample, "_infer_cnv.tsv"),
#                                     delim="\t",
#                                     gene_order_file="/expanse/lustre/projects/csd728/m7huang/single_cell_paik/infer_cnv/gene_name_ordering_file.txt",
#                                     ref_group_names=c("normal_EVT", "normal_CTB"))


# # perform infercnv operations to reveal cnv signal
# infercnv_obj_hmm = infercnv::run(infercnv_obj,
#                              cutoff=0.1,  # use 1 for smart-seq, 0.1 for 10x-genomics
#                              out_dir=out_dir,  # dir is auto-created for storing outputs
#                              cluster_by_groups=T,   # cluster
#                              denoise=T,
#                              HMM=T
#                              )
# saveRDS(infercnv_obj_hmm, "infer_cnv/infercnv_obj_hmm.RDS")

In [ ]:
column_names <- c(
  "proportion_scaled_dupli_chr1", "proportion_scaled_dupli_chr2", "proportion_scaled_dupli_chr3",
  "proportion_scaled_dupli_chr4", "proportion_scaled_dupli_chr5", "proportion_scaled_dupli_chr6",
  "proportion_scaled_dupli_chr7", "proportion_scaled_dupli_chr8", "proportion_scaled_dupli_chr9",
  "proportion_scaled_dupli_chr10", "proportion_scaled_dupli_chr11", "proportion_scaled_dupli_chr12",
  "proportion_scaled_dupli_chr13", "proportion_scaled_dupli_chr14", "proportion_scaled_dupli_chr15",
  "proportion_scaled_dupli_chr16", "proportion_scaled_dupli_chr17", "proportion_scaled_dupli_chr18",
  "proportion_scaled_dupli_chr19", "proportion_scaled_dupli_chr20", "proportion_scaled_dupli_chr21",
  "proportion_scaled_dupli_chr22"
)

troph$proportion_scaled_dupli_all <- rowSums(troph@meta.data[, column_names])/22

In [ ]:
write.csv(troph[[]], paste0(data_dir, "meta/troph_cnv_20260920.csv"))

# Monopogen (trophoblast)

In [ ]:
troph <- readRDS(file=file.path(paste0(data_dir, 'paik.harmony.combined.rds')))

In [ ]:
setwd("/expanse/lustre/projects/csd728/m7huang/single_cell_paik")

In [ ]:
library(data.table)

vcf_file <- "/expanse/lustre/projects/csd728/m7huang/iPSC_cell_line_whole_genome/tools/af-only-gnomad.hg38.vcf.gz"
vcf_germline <- fread(vcf_file, skip = "#CHROM")

In [ ]:
head(vcf_germline)

In [ ]:
troph <- readRDS(file=file.path(paste0(data_dir, 'paik.troph.harmony.combined.rds')))

In [ ]:
samples <-c('GSM8817243', 'SRR32499787', 'GSM8817241', 'SRR32499804',
            'SRR32499806', 'SRR32499808', 'SRR32499810')               
for (sample in samples){
    print(sample)
    get_variant_db(sample)
}

In [ ]:
get_variant_db <- function(sample){
    troph_combined <- subset(troph, orig.ident == sample)
    troph_combined <- RenameCells(troph_combined, new.names = gsub(paste0(sample, "_"), "", Cells(troph_combined)))
    
    cells_df <- as.data.frame(Cells(troph_combined))
    colnames(cells_df) <- c("cell_barcode")
    
    all_chrom <- c("chr1","chr2","chr3","chr4","chr5","chr6","chr7","chr8","chr9","chr10",
                    "chr11","chr12","chr13","chr14","chr15","chr16","chr17","chr18","chr19","chr20",
                      "chr21","chr22")
    
    for (chr in all_chrom){
        results <- get_variant_counts(chr, sample, troph_combined)
        cells_df <- cells_df %>%
          left_join(results, by = "cell_barcode")  
    }
    
    write.csv(cells_df, paste0(data_dir, "monopogen/", sample, ".cb.count.csv"))
}

In [ ]:
# this function gets the variants for each sample per chromosome
get_variant_counts <- function(chrom, sample, troph_combined){
    bm_sample <- paste0("bm_", sample)

    ## quality filter variants
    chr_filt <- read.csv(paste0("monopogen_results/results/", bm_sample, "/somatic/",
                             chrom, ".putativeSNVs.csv"))
    chr_filt <- subset(chr_filt, SVM_pos_score > 0.1 & 
                               0.1 < BAF_alt & BAF_alt < 0.5 & Depth_ref > 5 & Depth_alt > 5)
    chr_filt <- subset(chr_filt, LDrefine_merged_score > 0.25 | is.na(LDrefine_merged_score))

    ## filter germline variants 
    germ_chr <- subset(vcf_germline, `#CHROM` == chrom)
    colnames(germ_chr) <- c("chr", "pos", "ID", "Ref_allele", "Alt_allele", "QUAL", "FILTER", "INFO")
    non_overlap <- anti_join(chr_filt, germ_chr, by = c("chr", "pos", "Ref_allele", "Alt_allele"))
    colnames(non_overlap)[1:4] <- c("chr", "pos", "ref", "alt")

    # join cell barcode and kept variants
    mr_chr <- readRDS(paste0("monopogen_results/results/", bm_sample, "/somatic/", chrom, ".SNV_mat.RDS"))
    final_chr <- inner_join(mr_chr, non_overlap, by = c("chr", "pos", "ref", "alt"))

    # only keep cell barcodes filtered through QC
    final <- final_chr[, colnames(final_chr)[colnames(final_chr) %in% Cells(troph_combined)]]

    # sum the number of cell barcodes 
    results_vector <- sapply(final, 
                         function(x) sum(as.numeric(sub(".*/", "", x)) > 0))
    results_df <- data.frame(cell_barcode = names(results_vector), count = as.numeric(results_vector))
    colnames(results_df) <- c("cell_barcode", paste(chrom, sample, sep = "_"))
    results_df
}

In [ ]:
# # spot check:
# sample <- "bm_SRR32499787"
# sample2 <- "SRR32499787"
# chrom <- "chr18"

# samp_combined <- subset(troph, orig.ident == sample2)
# samp_combined <- RenameCells(samp_combined, new.names = gsub(paste0(sample2, "_"), "", Cells(samp_combined)))

# # ## quality filter variants
# chr_filt <- read.csv(paste0("monopogen_results/results/", sample, "/somatic/",
#                              chrom, ".putativeSNVs.csv"))
# chr_filt <- subset(chr_filt, SVM_pos_score > 0.1 & 
#                                0.1 < BAF_alt & BAF_alt < 0.5 & Depth_ref > 5 & Depth_alt > 5)
# chr_filt <- subset(chr_filt, LDrefine_merged_score > 0.25 | is.na(LDrefine_merged_score))


# ## filter germline variants 
# germ_chr <- subset(vcf_germline, `#CHROM` == chrom)
# colnames(germ_chr) <- c("chr", "pos", "ID", "Ref_allele", "Alt_allele", "QUAL", "FILTER", "INFO")

# overlap <- merge(chr_filt[,c("chr", "pos", "Ref_allele", "Alt_allele")], 
#              germ_chr[, c("chr", "pos", "Ref_allele", "Alt_allele")])
# non_overlap <- anti_join(chr_filt, germ_chr, by = c("chr", "pos", "Ref_allele", "Alt_allele"))
# colnames(non_overlap)[1:4] <- c("chr", "pos", "ref", "alt")

# dim(chr_filt)
# dim(overlap)
# dim(non_overlap)

# # join cell barcode and kept variants
# mr_chr <- readRDS(paste0("monopogen_results/results/", sample, "/somatic/", chrom, ".SNV_mat.RDS"))
# final_chr <- inner_join(mr_chr, non_overlap, by = c("chr", "pos", "ref", "alt"))



In [ ]:
# print(final_chr$AAAGGGCTCAGAGCAG)
# print(sum(as.numeric(sub(".*/", "", final_chr$AAAGGGCTCAGAGCAG)) > 0))

# subset(read.csv("outputs/monopogen/SRR32499787.cb.count.csv"), cell_barcode == "AAAGGGCTCAGAGCAG")$chr18_SRR32499787

In [ ]:
setwd("/expanse/lustre/projects/csd728/m7huang/single_cell_paik")

In [ ]:
troph <- readRDS(file=file.path(paste0(data_dir, 'DP_onlypaik.UCell.troph1.20260920.rds')))

In [ ]:
files <- unique(troph$orig.ident)
files

In [ ]:
all_cb <- NULL 

for (samp_name in files){
    cb_mat <- read.csv(paste0("outputs_20260919/DP_only/monopogen/", samp_name, ".cb.count.csv"), row.names = 1)
    cb_mat$num_mutations <- rowSums(cb_mat[2:23], na.rm = TRUE)
    cb_mat$ident <- paste(samp_name, cb_mat$cell_barcode, sep = "_")
    
    all_cb <- bind_rows(all_cb, cb_mat[,c("ident", "num_mutations")])
}
rownames(all_cb) <- all_cb$ident

In [ ]:
all_coverage <- NULL 

for (samp_name in files){
    coverage <- read.csv(paste0("cell_barcodes/", samp_name, "/coverage/", samp_name, ".allcells_coverage.txt"), 
                            sep = "\t")
    coverage$cell_barcode <- sapply(strsplit(as.character(coverage$File_Name), "_"), `[`, 1)
    coverage$ident <- paste0(samp_name, "_", coverage$cell_barcode)
    
    all_coverage <- bind_rows(all_coverage, coverage[,c("ident", "Coverage")])
}

rownames(all_coverage) <- all_coverage$ident

In [ ]:
troph <- AddMetaData(troph, metadata = all_cb)
troph <- AddMetaData(troph, metadata = all_coverage)

In [ ]:
hg38_num_bases = 3088286401
troph@meta.data['Number_of_bases_per106.megabase'] = troph@meta.data['Coverage']*100*hg38_num_bases/10**6
troph@meta.data['eMB'] = troph@meta.data['num_mutations']*100/troph@meta.data['Number_of_bases_per106.megabase']

In [ ]:
# meta <- troph@meta.data
# # meta$cell_type <- ifelse(meta$harmony_clusters == 6, "Extravillous trophoblast",
# #                  ifelse(meta$harmony_clusters == 7, "Cytotrophoblast", meta$harmony_clusters))
# meta$sample_description <- ifelse(meta$sample_description == "full term birh", "NTT-N",
#                  ifelse(meta$sample_description == "preterm birth", "EP-PE", meta$sample_description))
# head(meta)

In [ ]:
meta <- troph@meta.data

In [ ]:
summary(meta$eMB)

In [ ]:
write.csv(meta, paste0(data_dir, "meta/troph_eMB_20260920.csv"))

# Monopogen (all cells)

In [ ]:
setwd("/expanse/lustre/projects/csd728/m7huang/single_cell_paik")

In [ ]:
library(data.table)

vcf_file <- "/expanse/lustre/projects/csd728/m7huang/iPSC_cell_line_whole_genome/tools/af-only-gnomad.hg38.vcf.gz"
vcf_germline <- fread(vcf_file, skip = "#CHROM")

In [ ]:
head(vcf_germline)

In [ ]:
get_variant_db <- function(sample){
    troph_combined <- subset(all_cells, orig.ident == sample)
    troph_combined <- RenameCells(troph_combined, new.names = gsub(paste0(sample, "_"), "", Cells(troph_combined)))
    
    cells_df <- as.data.frame(Cells(troph_combined))
    colnames(cells_df) <- c("cell_barcode")
    
    all_chrom <- c("chr1","chr2","chr3","chr4","chr5","chr6","chr7","chr8","chr9","chr10",
                    "chr11","chr12","chr13","chr14","chr15","chr16","chr17","chr18","chr19","chr20",
                      "chr21","chr22")
    
    for (chr in all_chrom){
        results <- get_variant_counts(chr, sample, troph_combined)
        cells_df <- cells_df %>%
          left_join(results, by = "cell_barcode")  
    }
    
    write.csv(cells_df, paste0(data_dir, "monopogen/", sample, ".cb.count.csv"))
}

In [ ]:
# this function gets the variants for each sample per chromosome
get_variant_counts <- function(chrom, sample, troph_combined){
    bm_sample <- paste0("bm_", sample)

    ## quality filter variants
    chr_filt <- read.csv(paste0("monopogen_results/results/", bm_sample, "/somatic/",
                             chrom, ".putativeSNVs.csv"))
    chr_filt <- subset(chr_filt, SVM_pos_score > 0.1 & 
                               0.1 < BAF_alt & BAF_alt < 0.5 & Depth_ref > 5 & Depth_alt > 5)
    chr_filt <- subset(chr_filt, LDrefine_merged_score > 0.25 | is.na(LDrefine_merged_score))

    ## filter germline variants 
    germ_chr <- subset(vcf_germline, `#CHROM` == chrom)
    colnames(germ_chr) <- c("chr", "pos", "ID", "Ref_allele", "Alt_allele", "QUAL", "FILTER", "INFO")
    non_overlap <- anti_join(chr_filt, germ_chr, by = c("chr", "pos", "Ref_allele", "Alt_allele"))
    colnames(non_overlap)[1:4] <- c("chr", "pos", "ref", "alt")

    # join cell barcode and kept variants
    mr_chr <- readRDS(paste0("monopogen_results/results/", bm_sample, "/somatic/", chrom, ".SNV_mat.RDS"))
    final_chr <- inner_join(mr_chr, non_overlap, by = c("chr", "pos", "ref", "alt"))

    # only keep cell barcodes filtered through QC
    final <- final_chr[, colnames(final_chr)[colnames(final_chr) %in% Cells(troph_combined)]]

    # sum the number of cell barcodes 
    results_vector <- sapply(final, 
                         function(x) sum(as.numeric(sub(".*/", "", x)) > 0))
    results_df <- data.frame(cell_barcode = names(results_vector), count = as.numeric(results_vector))
    colnames(results_df) <- c("cell_barcode", paste(chrom, sample, sep = "_"))
    results_df
}

In [ ]:
samples <- c('SRR32499786','SRR32499787','SRR32499788','SRR32499789','SRR32499790','SRR32499791',
            'SRR32499792','SRR32499793','SRR32499794','SRR32499795','SRR32499796','SRR32499797',
            'SRR32499798','SRR32499799','SRR32499800','SRR32499801','SRR32499802','SRR32499803',
            'SRR32499804','SRR32499805','SRR32499806','SRR32499807','SRR32499808','SRR32499809',
            'SRR32499810','SRR32499811','SRR32499812','SRR32499813','SRR32499814','SRR32499815',
            'SRR32499816','SRR32499817')                  
for (sample in samples){
    print(sample)
    get_variant_db(sample)
}

In [ ]:
combined <- readRDS(file.path(paste0(data_dir, 'DP_onlypaik.harmony.combined.20260920.rds')))

In [ ]:
meta <- read.table("/expanse/lustre/projects/csd728/m7huang/single_cell_paik/data/20260919_SraRunTable.csv", sep = ",", header = TRUE)
colnames(meta)[colnames(meta) == "Run"] <- "orig.ident"
colnames(meta)

In [ ]:
matched <- meta[match(combined@meta.data$orig.ident, meta$orig.ident), ]
combined <- AddMetaData(combined, metadata = matched)

In [ ]:
files <- unique(combined$orig.ident)
files

In [ ]:
all_cb <- NULL 

for (samp_name in files){
    cb_mat <- read.csv(paste0("outputs_20260919/DP_only/monopogen_all/", samp_name, ".cb.count.csv"), row.names = 1)
    cb_mat$num_mutations <- rowSums(cb_mat[2:23], na.rm = TRUE)
    cb_mat$ident <- paste(samp_name, cb_mat$cell_barcode, sep = "_")
    
    all_cb <- bind_rows(all_cb, cb_mat[,c("ident", "num_mutations")])
}
rownames(all_cb) <- all_cb$ident

In [ ]:
all_coverage <- NULL 

for (samp_name in files){
    coverage <- read.csv(paste0("cell_barcodes/", samp_name, "/coverage/", samp_name, ".allcells_coverage.txt"), 
                            sep = "\t")
    coverage$cell_barcode <- sapply(strsplit(as.character(coverage$File_Name), "_"), `[`, 1)
    coverage$ident <- paste0(samp_name, "_", coverage$cell_barcode)
    
    all_coverage <- bind_rows(all_coverage, coverage[,c("ident", "Coverage")])
}

rownames(all_coverage) <- all_coverage$ident

In [ ]:
combined <- AddMetaData(combined, metadata = all_cb)
combined <- AddMetaData(combined, metadata = all_coverage)

In [ ]:
hg38_num_bases = 3088286401
combined@meta.data['Number_of_bases_per106.megabase'] = combined@meta.data['Coverage']*100*hg38_num_bases/10**6
combined@meta.data['eMB'] = combined@meta.data['num_mutations']*100/combined@meta.data['Number_of_bases_per106.megabase']

In [ ]:
combined$cell_type <- Idents(combined)

In [ ]:
meta <- combined@meta.data
meta$disease <- ifelse(meta$sample_description == "full term birh", "NTT-N",
                 ifelse(meta$sample_description == "preterm birth", "EP-PE", meta$sample_description))
head(meta)

In [ ]:
summary(meta$eMB)

In [ ]:
write.csv(meta, paste0(data_dir, "meta/meta_all_eMB.csv"))

# LMM

In [ ]:
library(lme4)
library("emmeans")

In [ ]:
setwd("/expanse/lustre/projects/csd728/m7huang/single_cell_paik/outputs_20260919/DP_only/meta")

## eMB

In [ ]:
df <- read.csv("meta_all_eMB.csv")

In [ ]:
df$scale_eMB <- log(1 + df$eMB)

In [ ]:
m <- lmer(scale_eMB ~ cell_type + (1 | orig.ident), data = df)
print(summary(m))

emm <- emmeans(m, ~ cell_type)

pairs(emm)

In [ ]:
options(repr.plot.width=6, repr.plot.height=6)

hist(
  residuals(m),
  breaks = 30,
  main = "Distribution of model residuals",
  xlab = "Residuals"
)

In [ ]:
m <- lmer(
  scale_eMB ~ cell_type * disease + (1 | orig.ident),
  data = df
)

print(summary(m))
emm_disease <- emmeans(m, ~ disease | cell_type)

pairs(emm_disease)

In [ ]:
hist(
  residuals(m),
  breaks = 30,
  main = "Distribution of model residuals",
  xlab = "Residuals"
)

## CNV

In [ ]:
df <- read.csv("troph_cnv_20260920.csv")

In [ ]:
m <- lmer(
  proportion_scaled_dupli_all ~ cell_type * disease + (1 | orig.ident),
  data = df
)

print(summary(m))
emm_disease <- emmeans(m, ~ disease | cell_type)

pairs(emm_disease)

In [ ]:
hist(
  residuals(m),
  breaks = 30,
  main = "Distribution of model residuals",
  xlab = "Residuals"
)

## Hypoxia

In [ ]:
df <- read.csv("troph_UCell_20260920.csv")

In [ ]:
m <- lmer(
  signature_1 ~ cell_type * disease + (1 | orig.ident),
  data = df
)

print(summary(m))
emm_disease <- emmeans(m, ~ disease | cell_type)

pairs(emm_disease)

In [ ]:
hist(
  residuals(m),
  breaks = 30,
  main = "Distribution of model residuals",
  xlab = "Residuals"
)